# Tutorial: Plan 004 Query Grammar Playground

**Audience:** Python beginners who know variables, functions, and lists.

**Goal:** learn how small Python objects can describe a query without executing it immediately.

We will build a deliberately tiny, in-memory model. It is a **conceptual playground**, not Archiver's promised public API.

**Outline:**
1. Make data objects with `@dataclass`.
2. Represent fields and predicates as objects.
3. build immutable queries with `replace()`.
4. Execute a query against toy rows.
5. Try one exercise.


## 1. A dataclass holds related values

`@dataclass` asks Python to generate useful methods for a class, including:

- `__init__` for constructing objects;
- `__repr__` for readable notebook output;
- `__eq__` for value-based equality.

Type annotations such as `path: str` name the fields. `frozen=True` blocks normal field assignment. `slots=True` prevents accidental new attributes and can reduce memory use.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, replace


@dataclass(frozen=True, slots=True)
class ToyFile:
    path: str
    size_bytes: int
    kind: str


files = (
    ToyFile("reports/final.pdf", 120, "document"),
    ToyFile("notes.txt", 5, "text"),
    ToyFile("reports/draft.md", 40, "text"),
    ToyFile("photos/cat.jpg", 800, "image"),
)

changed_copy = replace(files[0], size_bytes=125)
{
    "first object": files[0],
    "value equality": files[0] == ToyFile("reports/final.pdf", 120, "document"),
    "changed copy": changed_copy,
    "original unchanged": files[0].size_bytes,
}


{'first object': ToyFile(path='reports/final.pdf', size_bytes=120, kind='document'),
 'value equality': True,
 'changed copy': ToyFile(path='reports/final.pdf', size_bytes=125, kind='document'),
 'original unchanged': 120}

`replace(object, field=value)` creates another dataclass instance. It does not edit the original.

Dataclasses are not automatically immutable: `frozen=True` supplies that rule, and it is still best to place immutable values such as tuples inside them. This pattern lets a query object be safely reused to create several variations.


In [4]:
changed_copy == files[0]
changed_copy.kind

'document'

## 2. Fields and predicates are objects

A **field** identifies a value that can be queried. A **predicate** records a test we want to perform.

Calling `SIZE.at_least(100)` will construct a description of a test. It will not inspect any files yet.


In [5]:
@dataclass(frozen=True, slots=True)
class Field:
    name: str

    def equals(self, expected: object) -> Predicate:
        return Predicate(self, "equals", expected)

    def contains(self, expected: str) -> Predicate:
        return Predicate(self, "contains", expected)

    def at_least(self, expected: int) -> Predicate:
        return Predicate(self, "at_least", expected)


@dataclass(frozen=True, slots=True)
class Predicate:
    field: Field
    operator: str
    expected: object


PATH = Field("path")
SIZE = Field("size_bytes")
KIND = Field("kind")

path_filter = PATH.contains("reports/")
size_filter = SIZE.at_least(100)
kind_filter = KIND.equals("text")

path_filter, size_filter, kind_filter


(Predicate(field=Field(name='path'), operator='contains', expected='reports/'),
 Predicate(field=Field(name='size_bytes'), operator='at_least', expected=100),
 Predicate(field=Field(name='kind'), operator='equals', expected='text'))

In [10]:
# nested field object could generate another identical predicate.
size_filter.field.at_least(100) == size_filter

True

In [7]:
def describe(predicate: Predicate) -> str:
    return f"{predicate.field.name} {predicate.operator} {predicate.expected!r}"


predicate_box = [path_filter, size_filter, kind_filter]
{
    "stored in a list": [describe(item) for item in predicate_box],
    "nested field object": size_filter.field,
    "compared by value": size_filter == SIZE.at_least(100),
}


{'stored in a list': ["path contains 'reports/'",
  'size_bytes at_least 100',
  "kind equals 'text'"],
 'nested field object': Field(name='size_bytes'),
 'compared by value': True}

## 3. A query is an immutable description

Our `Query` stores predicates in a tuple. Its `where()` method uses `replace()` to return a new query containing one more predicate.

This is the key idea: building a query creates data. It does not execute the query or mutate the previous query.


In [11]:
@dataclass(frozen=True, slots=True)
class Query:
    source: str
    predicates: tuple[Predicate, ...] = ()

    def where(self, predicate: Predicate) -> Query:
        return replace(self, predicates=(*self.predicates, predicate))


all_files = Query(source="toy files")
large_files = all_files.where(SIZE.at_least(100))
large_reports = large_files.where(PATH.contains("reports/"))

{
    "original": all_files,
    "one condition": large_files,
    "two conditions": large_reports,
    "original still has no predicates": all_files.predicates,
}


{'original': Query(source='toy files', predicates=()),
 'one condition': Query(source='toy files', predicates=(Predicate(field=Field(name='size_bytes'), operator='at_least', expected=100),)),
 'two conditions': Query(source='toy files', predicates=(Predicate(field=Field(name='size_bytes'), operator='at_least', expected=100), Predicate(field=Field(name='path'), operator='contains', expected='reports/'))),
 'original still has no predicates': ()}

## 4. Execution is a separate job

The query objects only describe intent. The functions below interpret that description against our tuple of `ToyFile` rows.

`getattr(row, name)` reads an attribute whose name is stored in an object. `all(...)` requires every predicate to match.


In [30]:
# What it does: Dynamically reads the value from the file object.

# define a prediate
SIZE = Field("size_bytes")
size_filter = SIZE.at_least(100)
# define a file
File1 = ToyFile("reports/final.pdf", 120, "document")

# How it works:
# predicate.field.name extracts the string name of the attribute to look up (e.g., "size_bytes", "path", or "kind").
print(f"predicate.field.name = {size_filter.field.name}")

#getattr(row, "size_bytes") is equivalent to writing row.size_bytes, but allows the attribute name to be determined dynamically at runtime.
assert getattr(File1, "size_bytes")  == getattr(File1, size_filter.field.name)
print(f"The file size is : {getattr(File1, size_filter.field.name)}")
print(f"The expcted File size in the predicate: {size_filter.expected}")


predicate.field.name = size_bytes
The file size is : 120
The expcted File size in the predicate: 100


In [39]:
def matches(row: ToyFile, predicate: Predicate) -> bool:
    # get the value of the field in the row
    actual = getattr(row, predicate.field.name)

    if predicate.operator == "equals":
        return actual == predicate.expected
    if predicate.operator == "contains":
        return str(predicate.expected) in str(actual)
    if predicate.operator == "at_least":
        return int(actual) >= int(predicate.expected)

    raise ValueError(f"Unknown operator: {predicate.operator}")

def matches_all(query: Query, row: ToyFile) -> bool:
    """Returns True if the row satisfies all predicates in the query, False otherwise."""
    return all(matches(row, predicate) for predicate in query.predicates)
    
def execute(query: Query, rows: tuple[ToyFile, ...]) -> tuple[ToyFile, ...]:
    return tuple(
        row
        for row in rows
        if matches_all(query, row))


def dryrun(query: Query, rows: tuple[ToyFile, ...]) -> tuple[bool, ...]:
    return tuple(matches_all(query, row) for row in rows)



In [48]:
# files from the database
files = (
    ToyFile("reports/final.pdf", 120, "document"),
    ToyFile("notes.txt", 5, "text"),
    ToyFile("reports/draft.md", 40, "text"),
    ToyFile("photos/cat.jpg", 800, "image"),
)

# Queries
all_files = Query(source="toy files")
large_files = all_files.where(SIZE.at_least(20))
large_reports = large_files.where(PATH.contains("reports/"))

print(f"dryrun: {dryrun(large_reports, files)}")
print('Actual files:')
print(execute(large_reports, files))

# show a different predicate
print('\nlarge file predicate without the contains(report)')
execute(large_files, files)


dryrun: (True, False, True, False)
Actual files:
(ToyFile(path='reports/final.pdf', size_bytes=120, kind='document'), ToyFile(path='reports/draft.md', size_bytes=40, kind='text'))

large file predicate without the contains(report)


(ToyFile(path='reports/final.pdf', size_bytes=120, kind='document'),
 ToyFile(path='reports/draft.md', size_bytes=40, kind='text'),
 ToyFile(path='photos/cat.jpg', size_bytes=800, kind='image'))

## Exercise

Build a query for text files whose path contains `reports/`.

Before running the solution, predict:

1. Which row will match?
2. Will `all_files` change?
3. How many predicates will the new query contain?


In [ ]:
# Solution — try writing this yourself before running the cell.
text_reports = (
    all_files
    .where(PATH.contains("reports/"))
    .where(KIND.equals("text"))
)
answer = execute(text_reports, files)

assert tuple(item.path for item in answer) == ("reports/draft.md",)
assert all_files.predicates == ()
assert len(text_reports.predicates) == 2

answer


## What maps to Archiver?

- `Field`, `Predicate`, and `Query` are **descriptions of intent**.
- `where()` creates a new description with `replace()`.
- `execute()` interprets the description; a later Archiver layer will instead compile analogous typed objects into parameterized SQL and materialize domain objects.

### Pitfalls

- `@dataclass` alone does not make an object immutable; that is what `frozen=True` requests.
- A frozen dataclass can still contain a mutable list, so this query uses a tuple.
- The toy executor trusts string operator names and `getattr()`. Production code needs tighter validation and a compiler boundary.
- These names and method signatures are teaching aids, not promised Archiver APIs.

### Try next

Add a `smaller_than()` field method and teach `matches()` its operator. Notice that extending the vocabulary means changing both the description side and the execution side.
